# Producer C — Camera Events (Camera 3)

## Task 2.1.1: Kafka Producer Implementation

This notebook implements **Producer C** for the AWAS traffic monitoring system.  
It reads `camera_event_C.csv`, groups records by `batch_id`, and publishes each batch  
to the Kafka topic **`camera-events-C`** with a configurable delay between batches.

| Parameter | Value | Purpose |
|---|---|---|
| Topic | `camera-events-C` | Kafka topic for Camera 3 events |
| Producer ID | `C` | Metadata tag for source traceability |
| Batch sleep | 5 seconds | Simulates real-time event arrival rate |
| Bootstrap server | `host.docker.internal:9092` | Kafka container reachable from inside pyspark container via Docker host |
| Message key | `car_plate` | Enables Spark to co-partition events by vehicle |

### Workflow
1. Load `camera_event_C.csv` into a DataFrame  
2. Sort by `batch_id` and group records per batch  
3. For each batch: publish all records to Kafka, then sleep `BATCH_SLEEP_SECONDS`  
4. Each message payload includes all CSV fields plus `producer_id = "C"` for traceability

In [ ]:
# Install required libraries (run once if not already installed)
# !pip install kafka3 pandas

In [ ]:
import pandas as pd
from json import dumps
from time import sleep
from kafka3 import KafkaProducer   # kafka-python (pip install kafka-python)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
HOST_IP             = "host.docker.internal"
KAFKA_PORT          = 9092
TOPIC               = "camera-events-C"
PRODUCER_ID         = "C"
BATCH_SLEEP_SECONDS = 5

import os
BASE_DIR = "/home/student/Assignment 2/FIT3182"
CSV_PATH = os.path.join(BASE_DIR, "data", "camera_event_C.csv")
# ─────────────────────────────────────────────────────────────────────────────

print(f"Bootstrap server : {HOST_IP}:{KAFKA_PORT}")
print(f"Topic            : {TOPIC}")
print(f"Producer ID      : {PRODUCER_ID}")
print(f"CSV path         : {CSV_PATH}")


In [ ]:
# Load camera event data and preview
df = pd.read_csv(CSV_PATH)

df["batch_id"]      = df["batch_id"].astype(int)
df["camera_id"]     = df["camera_id"].astype(int)
df["speed_reading"] = df["speed_reading"].astype(float)

print(f"Loaded {len(df)} records across {df['batch_id'].nunique()} batches")
print(f"Batch IDs: {sorted(df['batch_id'].unique().tolist())}\n")
df.head()

In [ ]:
def connect_kafka_producer():
    """
    Create and return a KafkaProducer connected to the configured bootstrap server.
    Messages are serialised to JSON-encoded ASCII bytes.
    Returns None if the connection fails.
    """
    producer = None
    try:
        producer = KafkaProducer(
            bootstrap_servers=[f"{HOST_IP}:{KAFKA_PORT}"],
            value_serializer=lambda x: dumps(x).encode("ascii"),
            api_version=(0, 10)
        )
        print(f"Connected to Kafka at {HOST_IP}:{KAFKA_PORT}")
    except Exception as ex:
        print(f"Exception while connecting to Kafka: {ex}")
    return producer

In [ ]:
def publish_message(producer, topic, key, data):
    """
    Publish a single message to the given Kafka topic.

    Args:
        producer : KafkaProducer instance
        topic    : Kafka topic name (str)
        key      : Message key as string — used by Kafka for partition routing
        data     : Message payload (dict) — serialised by producer's value_serializer
    """
    try:
        key_bytes = bytes(key, encoding="utf-8")
        producer.send(topic, key=key_bytes, value=data)
        producer.flush()
        print(f"  Published: {data}")
    except Exception as ex:
        print(f"  WARNING — failed to publish record (key={key}): {ex}")

In [ ]:
if __name__ == "__main__":
    producer = connect_kafka_producer()

    if producer is None:
        print("ERROR: Could not connect to Kafka. Check HOST_IP and that the Kafka container is running.")
    else:
        batches = sorted(df["batch_id"].unique())
        print(f"\nPublishing {len(df)} records across {len(batches)} batch(es) to topic '{TOPIC}'")
        print(f"Sleep between batches: {BATCH_SLEEP_SECONDS}s\n")

        for batch_id in batches:
            batch_df = df[df["batch_id"] == batch_id]
            print(f"── Batch {batch_id} ({len(batch_df)} records) ──────────────────────")

            for _, row in batch_df.iterrows():
                try:
                    event = {
                        "event_id":      row["event_id"],
                        "batch_id":      int(row["batch_id"]),
                        "car_plate":     row["car_plate"],
                        "camera_id":     int(row["camera_id"]),
                        "timestamp":     row["timestamp"],
                        "speed_reading": float(row["speed_reading"]),
                        "producer_id":   PRODUCER_ID
                    }
                    publish_message(producer, TOPIC, row["car_plate"], event)
                except Exception as ex:
                    print(f"  WARNING — skipping malformed row: {ex}")

            print(f"  Batch {batch_id} complete. Sleeping {BATCH_SLEEP_SECONDS}s...\n")
            sleep(BATCH_SLEEP_SECONDS)

        print("All batches published.")